In [ ]:
!pip -q install lightgbm hdbscan networkx joblib scipy

import os, glob, gc, re
import numpy as np
import pandas as pd

import lightgbm as lgb
import hdbscan
import networkx as nx
import joblib

from scipy.stats import ks_2samp
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score,
    average_precision_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import IsolationForest
from sklearn.calibration import CalibratedClassifierCV

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [ ]:
import numpy as np, pandas as pd, sklearn, lightgbm, hdbscan, networkx
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

numpy: 2.0.2
pandas: 2.2.2
sklearn: 1.6.1
lightgbm: 4.6.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_FOLDER = "/content/drive/MyDrive/dataset"
OUTPUT_DIR = "/content/ids_research_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_SIZE = 0.20
VAL_SIZE  = 0.10
MI_SAMPLE = 200000
TOP_K_MI_L1 = 20
TOP_K_MI_L2 = 20

ISO_SAMPLE = 120000
ISO_CONTAM = 0.01

HDBSCAN_MAX = 12000  # cluster only anomalies

print("Drive folder:", DRIVE_FOLDER)
print("Output dir:", OUTPUT_DIR)

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/dataset
Output dir: /content/ids_research_outputs


In [ ]:
csv_files = sorted(glob.glob(os.path.join(DRIVE_FOLDER, "*.csv")))
print("Found CSV files:", len(csv_files))
for f in csv_files[:10]:
    print(" -", os.path.basename(f))

dfs = []
for path in csv_files:
    temp = pd.read_csv(path, low_memory=False)
    temp["_source_file"] = os.path.basename(path)
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

print("Loaded df shape:", df.shape)
print("Columns:", len(df.columns))
print("Has Label?", "Label" in df.columns)
print("Has _source_file?", "_source_file" in df.columns)

Found CSV files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv
Loaded df shape: (2830743, 80)
Columns: 80
Has Label? True
Has _source_file? True


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2522362 entries, 0 to 2522361
Data columns (total 74 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df.describe()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Fwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522009e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06,2.522362e+06
mean,8.704762e+03,1.658132e+07,1.027627e+01,1.156596e+01,6.115751e+02,1.813315e+04,2.310918e+02,1.919464e+01,6.347010e+01,7.727759e+01,9.743700e+02,4.313467e+01,3.404133e+02,3.763118e+02,inf,inf,1.445246e+06,3.276120e+06,1.029310e+07,1.703157e+05,1.624176e+07,2.917447e+06,3.666366e+06,1.013665e+07,1.134967e+06,1.110336e+07,2.026477e+06,1.667647e+06,5.257358e+06,1.085438e+06,4.873805e-02,3.171630e-05,-2.918210e+04,-2.553644e+03,4.083435e+04,6.509475e+03,1.682349e+01,1.063098e+03,1.905415e+02,3.299988e+02,5.454386e+05,3.214685e-02,4.873805e-02,2.719673e-04,2.975192e-01,3.121875e-01,1.014276e-01,3.171630e-05,2.731567e-04,7.003495e-01,2.123115e+02,6.347010e+01,3.404133e+02,-2.918210e+04,1.027627e+01,6.115633e+02,1.156596e+01,1.813277e+04,7.265655e+03,2.230826e+03,6.005904e+00,-3.080307e+03,9.152169e+04,4.616313e+04,1.719104e+05,6.542300e+04,9.331578e+06,5.654433e+05,9.757716e+06,8.887157e+06
std,1.902507e+04,3.522426e+07,7.941738e+02,1.056594e+03,1.058499e+04,2.397434e+06,7.561625e+02,6.079447e+01,1.955015e+02,2.967953e+02,2.037859e+03,7.087022e+01,6.324238e+02,8.808314e+02,NaN,NaN,4.681883e+06,8.454525e+06,2.567868e+07,3.013372e+06,3.515781e+07,1.001355e+07,1.013934e+07,2.575975e+07,9.056733e+06,3.022113e+07,9.390609e+06,6.628389e+06,1.809655e+07,8.794528e+06,2.153199e-01,5.631635e-03,2.230271e+07,1.538422e+06,1.932083e+05,3.813312e+04,2.557873e+01,2.121197e+03,3.182915e+02,6.607343e+02,1.735993e+06,1.763900e-01,2.153199e-01,1.648919e-02,4.571669e-01,4.633860e-01,3.018941e-01,5.631635e-03,1.652520e-02,6.955667e-01,3.454353e+02,1.955015e+02,6.324238e+02,2.230271e+07,7.941738e+02,1.057067e+04,1.056594e+03,2.397401e+06,1.459869e+04,8.928420e+03,6.742059e+02,1.149402e+06,6.864412e+05,4.164568e+05,1.085243e+06,6.109712e+05,2.484157e+07,4.872678e+06,2.561067e+07,2.457481e+07
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.00

In [ ]:
print('correlation matrix')
df.select_dtypes(include=['number']).corr()

correlation matrix


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Fwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
Destination Port,1.000000,-0.171828,-0.004652,-0.004356,0.009611,-0.003388,0.090989,-0.047160,0.136916,0.123060,-0.212699,-0.262417,-0.238842,-0.190515,0.070287,0.359537,-0.113636,-0.131931,-0.130869,-0.023383,-0.171004,-0.107032,-0.111044,-0.129597,-0.055521,-0.119856,-0.075871,-0.041670,-0.061670,-0.055113,0.234993,0.004445,0.000457,0.000761,0.348251,0.143760,-0.255221,-0.143499,-0.171428,-0.139519,-0.077536,-0.036841,0.234993,-0.007134,-0.217887,0.585195,0.523381,0.004445,-0.007044,0.027562,-0.172832,0.136916,-0.238842,0.000457,-0.004652,0.009624,-0.004356,-0.003388,-0.133049,0.206231,-0.003543,0.001003,-0.040678,-0.048227,-0.058142,-0.027193,-0.127989,0.006732,-0.123677,-0.129539
Flow Duration,-0.171828,1.000000,0.020571,0.019389,0.063509,0.015998,0.263401,-0.115716,0.134033,0.224957,0.485858,-0.238692,0.448503,0.433671,-0.024982,-0.109900,0.534831,0.734884,0.779979,0.060566,0.998529,0.543119,0.721063,0.779716,0.209850,0.819887,0.408650,0.517722,0.580336,0.212795,-0.017047,-0.002651,-0.000097,-0.001049,-0.099488,-0.080348,-0.247385,0.501514,0.417241,0.445197,0.271962,0.217883,-0.017047,0.008075,0.179773,0.033256,-0.117224,-0.002651,0.008023,-0.165840,0.394800,0.134033,0.448503,-0.000097,0.020571,0.063592,0.019389,0.015998,0.094089,-0.034352,0.015728,-0.001236,0.184985,0.238497,0.289933,0.117259,0.764124,0.240369,0.775744,0.734123
Total Fwd Packets,-0.004652,0.020571,1.000000,0.999070,0.365510,0.996993,0.009071,-0.003255,-0.000262,0.001110,0.022552,-0.006068,0.021123,0.006045,0.000391,-0.002342,-0.001366,-0.000954,0.001862,-0.000589,0.020240,-0.001361,-0.000333,0.001418,-0.001290,0.023794,-0.000883,0.000475,0.003095,-0.001144,0.001882,-0.000059,0.000476,0.013750,-0.002132,-0.001655,-0.006792,0.021735,0.024239,0.011654,0.005141,-0.001337,0.001882,0.000339,0.007061,0.001378,-0.003794,-0.000059,0.000337,0.000882,0.021865,-0.000262,0.021123,0.000476,1.000000,0.365992,0.999070,0.996987,0.003729,-0.000658,0.887386,-0.000181,0.039822,0.008207,0.030318,0.041186,0.001402,0.000676,0.001483,0.001266
Total Backward Packets,-0.004356,0.019389,0.999070,1.000000,0.359457,0.994430,0.008764,-0.002834,-0.000617,0.000743,0.022342,-0.005343,0.021244,0.005722,0.000335,-0.002383,-0.001711,-0.001426,0.001406,-0.000615,0.019040,-0.001541,-0.000860,0.000947,-0.001160,0.023219,-0.001039,0.000272,0.003177,-0.001100,0.001673,-0.000061,0.000757,0.013766,-0.002231,-0.001369,-0.006152,0.021469,0.024655,0.011508,0.005111,-0.001183,0.001673,0.000214,0.006490,0.001203,-0.003151,-0.000061,0.000213,0.003279,0.022290,-0.000617,0.021244,0.000757,0.999070,0.359939,1.000000,0.994424,0.003167,-0.000660,0.882566,0.000021,0.038853,0.006318,0.028465,0.041185,0.001020,0.000364,0.001045,0.000938
Total Length of Fwd Packets,0.009611,0.063509,0.365510,0.359457,1.000000,0.353781,0.196242,-0.001737,0.185073,0.158876,0.020773,-0.027818

In [ ]:
required_cols = ["Label", "_source_file"]
for c in required_cols:
    assert c in df.columns, f"Missing column: {c}"

print("Rows:", len(df))
print("Files:", df["_source_file"].nunique())
print("Top labels:\n", df["Label"].value_counts().head(10))

Rows: 2830743
Files: 8
Top labels:
 Label
BENIGN              2273097
DoS Hulk             231073
PortScan             158930
DDoS                 128027
DoS GoldenEye         10293
FTP-Patator            7938
SSH-Patator            5897
DoS slowloris          5796
DoS Slowhttptest       5499
Bot                    1966
Name: count, dtype: int64


In [ ]:
LABEL_COL = "Label"
SOURCE_COL = "_source_file"

def map_attack_family(label):
    lbl = str(label).lower().strip().replace("�", " ")
    if lbl == "benign":
        return "BENIGN"
    if "portscan" in lbl:
        return "PortScan"
    if "ftp-patator" in lbl or "ssh-patator" in lbl or "patator" in lbl:
        return "BruteForce"
    if "web attack" in lbl or "xss" in lbl or "sql injection" in lbl:
        return "WebAttack"
    if "heartbleed" in lbl or "infiltration" in lbl:
        return "TrueRare"
    if "bot" in lbl:
        return "Bot"
    if "ddos" in lbl or lbl.startswith("dos") or "dos " in lbl:
        return "DoS"
    return "OtherAttack"

df["Level1_Label"] = df[LABEL_COL].apply(lambda x: "BENIGN" if str(x).strip().upper() == "BENIGN" else "ATTACK")
df["Level2_Family"] = df[LABEL_COL].apply(map_attack_family)

print(df["Level2_Family"].value_counts())
print("Files:", df[SOURCE_COL].nunique())

Level2_Family
BENIGN        2273097
DoS            380688
PortScan       158930
BruteForce      13835
WebAttack        2180
Bot              1966
TrueRare           47
Name: count, dtype: int64
Files: 8


In [ ]:
print("L1 counts:\n", df["Level1_Label"].value_counts())
print("\nL2 counts:\n", df["Level2_Family"].value_counts())

print("\nPer-file ATTACK family distribution:")
attack_only = df[df["Level1_Label"]=="ATTACK"]
print(attack_only.groupby("_source_file")["Level2_Family"].value_counts())

print("\nPortScan source coverage:")
print(df[df["Level2_Family"]=="PortScan"]["_source_file"].value_counts())

L1 counts:
 Level1_Label
BENIGN    2273097
ATTACK     557646
Name: count, dtype: int64

L2 counts:
 Level2_Family
BENIGN        2273097
DoS            380688
PortScan       158930
BruteForce      13835
WebAttack        2180
Bot              1966
TrueRare           47
Name: count, dtype: int64

Per-file ATTACK family distribution:
_source_file                                                 Level2_Family
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv             DoS              128027
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv         PortScan         158930
Friday-WorkingHours-Morning.pcap_ISCX.csv                    Bot                1966
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  TrueRare             36
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv       WebAttack          2180
Tuesday-WorkingHours.pcap_ISCX.csv                           BruteForce        13835
Wednesday-workingHours.pcap_ISCX.csv                         DoS              25266

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
before = len(df)
df = df.drop_duplicates(subset=num_cols + [LABEL_COL]).reset_index(drop=True)
after = len(df)
print("Removed duplicates:", before - after)

const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=const_cols, errors="ignore")
print("Dropped constant cols:", len(const_cols))

Removed duplicates: 308381
Dropped constant cols: 8


In [ ]:
X_num = df.select_dtypes(include=[np.number]).copy()
X_num.replace([np.inf, -np.inf], np.nan, inplace=True)

for c in ["Flow Bytes/s", "Flow Packets/s", "Fwd Packets/s", "Bwd Packets/s"]:
    if c in X_num.columns:
        X_num[c] = X_num[c].clip(lower=0)
        X_num[c] = np.log1p(X_num[c])

X_num = X_num.fillna(X_num.median(numeric_only=True))

# downcast
for c in X_num.columns:
    if pd.api.types.is_float_dtype(X_num[c]):
        X_num[c] = X_num[c].astype(np.float32)
    elif pd.api.types.is_integer_dtype(X_num[c]):
        X_num[c] = X_num[c].astype(np.int32)

scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=X_num.columns).astype(np.float32)

del X_num
gc.collect()

print("Feature matrix shape:", X_scaled.shape)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "robust_scaler.pkl"), compress=3)

Feature matrix shape: (2522362, 70)


['/content/ids_research_outputs/robust_scaler.pkl']

In [ ]:
def select_top_mi(X, y, base_features, top_k=20, sample_size=200000, seed=42):
    base_features = [f for f in base_features if f in X.columns]

    if len(X) > sample_size:
        idx = np.random.RandomState(seed).choice(X.index, size=sample_size, replace=False)
        Xs = X.loc[idx]
        ys = y.loc[idx]
    else:
        Xs, ys = X, y

    y_enc = pd.factorize(ys)[0]
    mi = mutual_info_classif(Xs, y_enc, random_state=seed)
    mi_s = pd.Series(mi, index=X.columns).sort_values(ascending=False)

    selected = list(dict.fromkeys(base_features + mi_s.head(top_k).index.tolist()))
    return selected, mi_s

In [ ]:
l1_base = [
    "Flow Duration","Total Fwd Packets","Total Backward Packets",
    "Total Length of Fwd Packets","Total Length of Bwd Packets",
    "Flow Bytes/s","Flow Packets/s","Fwd Packets/s","Bwd Packets/s",
    "Fwd Packet Length Mean","Bwd Packet Length Mean",
    "Packet Length Mean","Packet Length Std","Average Packet Size",
    "Fwd IAT Mean","Bwd IAT Mean","Fwd IAT Std","Bwd IAT Std",
    "ACK Flag Count","PSH Flag Count","Destination Port"
]
y_l1 = df["Level1_Label"]
l1_features, l1_mi = select_top_mi(X_scaled, y_l1, l1_base, top_k=TOP_K_MI_L1, sample_size=MI_SAMPLE, seed=RANDOM_STATE)

X_l1 = X_scaled[l1_features]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_l1, y_l1, test_size=TEST_SIZE, stratify=y_l1, random_state=RANDOM_STATE
)

# class weights
classes = np.array(["BENIGN","ATTACK"])
w = compute_class_weight("balanced", classes=classes, y=y_tr)
cw = {"BENIGN": w[0], "ATTACK": w[1]}

l1 = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=250, learning_rate=0.05,
    num_leaves=31, min_child_samples=100,
    subsample=0.8, colsample_bytree=0.8,
    class_weight=cw,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
l1.fit(X_tr, y_tr)

attack_idx = list(l1.classes_).index("ATTACK")
prob = l1.predict_proba(X_te)[:, attack_idx]
pred = np.where(prob >= 0.5, "ATTACK", "BENIGN")

print("=== OPTIMISTIC L1 (row-random) ===")
print(confusion_matrix(y_te, pred, labels=["BENIGN","ATTACK"]))
print(classification_report(y_te, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score((y_te=="ATTACK").astype(int), prob))
print("PR-AUC:", average_precision_score((y_te=="ATTACK").astype(int), prob))

joblib.dump(l1_features, os.path.join(OUTPUT_DIR, "l1_features.pkl"), compress=3)
joblib.dump(l1, os.path.join(OUTPUT_DIR, "l1_model.pkl"), compress=3)

=== OPTIMISTIC L1 (row-random) ===
[[418490    807]
 [    40  85136]]
              precision    recall  f1-score   support

      ATTACK       0.99      1.00      1.00     85176
      BENIGN       1.00      1.00      1.00    419297

    accuracy                           1.00    504473
   macro avg       1.00      1.00      1.00    504473
weighted avg       1.00      1.00      1.00    504473

ROC-AUC: 0.9999661318781935
PR-AUC: 0.9998246003994076


['/content/ids_research_outputs/l1_model.pkl']

In [ ]:
# Split train into train/val
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=VAL_SIZE, stratify=y_tr, random_state=RANDOM_STATE
)

l1.fit(X_tr2, y_tr2)

# calibrate on VAL (warning about prefit is okay)
cal_l1 = CalibratedClassifierCV(l1, method="isotonic", cv="prefit")
cal_l1.fit(X_val, y_val)

attack_idx = list(cal_l1.classes_).index("ATTACK")
val_prob = cal_l1.predict_proba(X_val)[:, attack_idx]
y_val_bin = (y_val=="ATTACK").astype(int).values

prec, rec, thr = precision_recall_curve(y_val_bin, val_prob)
target_recall = 0.95
valid = np.where(rec >= target_recall)[0]
if len(valid)==0:
    best_thr = 0.5
else:
    best_i = valid[np.argmax(prec[valid])]
    best_thr = float(thr[best_i-1] if best_i>0 else 0.0)

print("Chosen L1 threshold:", best_thr)

# Evaluate tuned threshold on test
te_prob = cal_l1.predict_proba(X_te)[:, attack_idx]
te_pred = np.where(te_prob >= best_thr, "ATTACK", "BENIGN")

print("=== L1 with calibrated probs + tuned threshold ===")
print(confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"]))
print(classification_report(y_te, te_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score((y_te=="ATTACK").astype(int), te_prob))
print("PR-AUC:", average_precision_score((y_te=="ATTACK").astype(int), te_prob))

joblib.dump(cal_l1, os.path.join(OUTPUT_DIR, "l1_calibrated.pkl"), compress=3)
joblib.dump(best_thr, os.path.join(OUTPUT_DIR, "l1_threshold.pkl"), compress=3)

/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


Chosen L1 threshold: 0.9770408163265306
=== L1 with calibrated probs + tuned threshold ===
[[419158    139]
 [  2821  82355]]
              precision    recall  f1-score   support

      ATTACK       1.00      0.97      0.98     85176
      BENIGN       0.99      1.00      1.00    419297

    accuracy                           0.99    504473
   macro avg       1.00      0.98      0.99    504473
weighted avg       0.99      0.99      0.99    504473

ROC-AUC: 0.9999464335333705
PR-AUC: 0.9997473689712688


['/content/ids_research_outputs/l1_threshold.pkl']

In [ ]:
attack_df = df[df["Level1_Label"]=="ATTACK"].copy()
supervised_families = ["DoS","PortScan","BruteForce","WebAttack","Bot"]
attack_df = attack_df[attack_df["Level2_Family"].isin(supervised_families)].copy()

l2_base = [
    "Flow Duration","Total Fwd Packets","Total Backward Packets",
    "Total Length of Fwd Packets","Total Length of Bwd Packets",
    "Flow Bytes/s","Flow Packets/s","Fwd Packets/s","Bwd Packets/s",
    "Fwd IAT Mean","Fwd IAT Std","Bwd IAT Mean","Bwd IAT Std",
    "Fwd Packet Length Mean","Bwd Packet Length Mean",
    "Packet Length Mean","Packet Length Std","Average Packet Size",
    "ACK Flag Count","PSH Flag Count","Destination Port"
]
y_l2 = attack_df["Level2_Family"]
l2_features, l2_mi = select_top_mi(X_scaled.loc[attack_df.index], y_l2, l2_base, top_k=TOP_K_MI_L2, sample_size=min(MI_SAMPLE,len(attack_df)), seed=RANDOM_STATE)

X_l2 = X_scaled.loc[attack_df.index, l2_features]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_l2, y_l2, test_size=TEST_SIZE, stratify=y_l2, random_state=RANDOM_STATE
)

classes = np.unique(y_tr)
w = compute_class_weight("balanced", classes=classes, y=y_tr)
cw2 = dict(zip(classes, w))

l2 = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=250, learning_rate=0.05,
    num_leaves=31, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    class_weight=cw2,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
)
l2.fit(X_tr, y_tr)

print("=== OPTIMISTIC L2 (row-random) ===")
pred = l2.predict(X_te)
print(confusion_matrix(y_te, pred, labels=list(l2.classes_)))
print(classification_report(y_te, pred, zero_division=0))

joblib.dump(l2_features, os.path.join(OUTPUT_DIR, "l2_features.pkl"), compress=3)
joblib.dump(l2, os.path.join(OUTPUT_DIR, "l2_model.pkl"), compress=3)

=== OPTIMISTIC L2 (row-random) ===
[[  391     0     0     0     0]
 [    0  1830     0     0     0]
 [    0     0 64323    12    18]
 [    0     0     6 18152     6]
 [    0     0     3     0   426]]
              precision    recall  f1-score   support

         Bot       1.00      1.00      1.00       391
  BruteForce       1.00      1.00      1.00      1830
         DoS       1.00      1.00      1.00     64353
    PortScan       1.00      1.00      1.00     18164
   WebAttack       0.95      0.99      0.97       429

    accuracy                           1.00     85167
   macro avg       0.99      1.00      0.99     85167
weighted avg       1.00      1.00      1.00     85167



['/content/ids_research_outputs/l2_model.pkl']

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

SUP_FAMS = ["DoS","PortScan","BruteForce","WebAttack","Bot"]
FILE_COL = "_source_file"

def loo_eval_L2_per_file(df, X_scaled, l2_features, file_col=FILE_COL, seed=42, min_train=5000):
    files = sorted(df[file_col].unique().tolist())
    results = []

    for test_file in files:
        # Test indices: supervised attack families only
        te_mask = (df[file_col] == test_file) & (df["Level1_Label"] == "ATTACK") & (df["Level2_Family"].isin(SUP_FAMS))
        te_idx = df.index[te_mask]

        if len(te_idx) == 0:
            results.append({
                "test_file": test_file,
                "train_attack_rows": 0,
                "test_attack_rows": 0,
                "classes_in_train": "",
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "note": "No supervised L2 attacks in test"
            })
            continue

        # Train indices: all other files, supervised families only
        tr_mask = (df[file_col] != test_file) & (df["Level1_Label"] == "ATTACK") & (df["Level2_Family"].isin(SUP_FAMS))
        tr_idx = df.index[tr_mask]

        if len(tr_idx) < min_train:
            results.append({
                "test_file": test_file,
                "train_attack_rows": int(len(tr_idx)),
                "test_attack_rows": int(len(te_idx)),
                "classes_in_train": "",
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "note": "Too few training samples"
            })
            continue

        y_tr = df.loc[tr_idx, "Level2_Family"]
        y_te = df.loc[te_idx, "Level2_Family"]

        classes = np.unique(y_tr)
        if len(classes) < 2:
            results.append({
                "test_file": test_file,
                "train_attack_rows": int(len(tr_idx)),
                "test_attack_rows": int(len(te_idx)),
                "classes_in_train": ",".join(sorted(classes)),
                "macro_f1": np.nan,
                "weighted_f1": np.nan,
                "note": "Train has <2 classes"
            })
            continue

        X_tr = X_scaled.loc[tr_idx, l2_features]
        X_te = X_scaled.loc[te_idx, l2_features]

        w = compute_class_weight("balanced", classes=classes, y=y_tr)
        cw = dict(zip(classes, w))

        model = lgb.LGBMClassifier(
            objective="multiclass",
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight=cw,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1
        )
        model.fit(X_tr, y_tr)

        pred = model.predict(X_te)

        macro = f1_score(y_te, pred, average="macro", zero_division=0)
        weighted = f1_score(y_te, pred, average="weighted", zero_division=0)

        support = y_te.value_counts().to_dict()

        results.append({
            "test_file": test_file,
            "train_attack_rows": int(len(tr_idx)),
            "test_attack_rows": int(len(te_idx)),
            "classes_in_train": ",".join(sorted(classes)),
            "macro_f1": float(macro),
            "weighted_f1": float(weighted),
            "support_DoS": int(support.get("DoS", 0)),
            "support_PortScan": int(support.get("PortScan", 0)),
            "support_BruteForce": int(support.get("BruteForce", 0)),
            "support_WebAttack": int(support.get("WebAttack", 0)),
            "support_Bot": int(support.get("Bot", 0)),
            "note": ""
        })

    return pd.DataFrame(results)

loo_l2_df = loo_eval_L2_per_file(df, X_scaled, l2_features, file_col=FILE_COL, seed=RANDOM_STATE, min_train=5000)

out_path = os.path.join(OUTPUT_DIR, "loo_l2_results.csv")
loo_l2_df.to_csv(out_path, index=False)
print("Saved:", out_path)

print("\nTop results:")
print(loo_l2_df.sort_values("macro_f1", ascending=False).head(10))

print("\nWorst results (non-NaN):")
print(loo_l2_df.dropna(subset=["macro_f1"]).sort_values("macro_f1").head(10))

loo_l2_df

Saved: /content/ids_research_outputs/loo_l2_results.csv

Top results:
                                           test_file  train_attack_rows  \
0   Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv             297815   
7               Wednesday-workingHours.pcap_ISCX.csv             232083   
1  Friday-WorkingHours-Afternoon-PortScan.pcap_IS...             335012   
2          Friday-WorkingHours-Morning.pcap_ISCX.csv             423878   
5  Thursday-WorkingHours-Morning-WebAttacks.pcap_...             423688   
6                 Tuesday-WorkingHours.pcap_ISCX.csv             416679   
3                  Monday-WorkingHours.pcap_ISCX.csv                  0   
4  Thursday-WorkingHours-Afternoon-Infilteration....                  0   

   test_attack_rows                       classes_in_train  macro_f1  \
0            128016  Bot,BruteForce,DoS,PortScan,WebAttack  0.235464   
7            193748  Bot,BruteForce,DoS,PortScan,WebAttack  0.032350   
1             90819           Bot,Brut

,test_file,train_attack_rows,test_attack_rows,classes_in_train,macro_f1,weighted_f1,support_DoS,support_PortScan,support_BruteForce,support_WebAttack,support_Bot,note
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,297815,128016,"Bot,BruteForce,DoS,PortScan,WebAttack",0.235464,0.706393,128016.0,0.0,0.0,0.0,0.0,
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,335012,90819,"Bot,BruteForce,DoS,WebAttack",0.000000,0.000000,0.0,90819.0,0.0,0.0,0.0,
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,423878,1953,"BruteForce,DoS,PortScan,WebAttack",0.000000,0.000000,0.0,0.0,0.0,0.0,1953.0,
3,Monday-WorkingHours.pcap_ISCX.csv,0,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No supervised L2 attacks in test
4,Thursday-WorkingHours-Afternoon-Infilteration....,0,0,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No supervised L2 attacks in test
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,423688,2143,"Bot,BruteForce,DoS,PortScan",0.000000,0.000000,0.0,0.0,0.0,2143.0,0.0,
6,Tuesday-WorkingHours.pcap_ISCX.csv,416679,9152,"Bot,DoS,PortScan,WebAttack",0.000000,0.000000,0.0,0.0,9152.0,0.0,0.0,
7,Wednesday-workingHours.pcap_ISCX.csv,232083,193748,"Bot,BruteForce,DoS,PortScan,WebAttack",0.032350,0.097049,193748.0,0.0,0.0,0.0,0.0,


In [ ]:
worst_file = loo_l2_df.dropna(subset=["macro_f1"]).sort_values("macro_f1").iloc[0]["test_file"]
print("Worst file:", worst_file)

Worst file: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


In [ ]:
iso_cols = [c for c in [
    "Flow Duration","Flow Bytes/s","Flow Packets/s",
    "Fwd Packets/s","Bwd Packets/s",
    "Packet Length Mean","Average Packet Size",
    "Fwd IAT Mean","Bwd IAT Mean","Destination Port"
] if c in X_scaled.columns]

ben_idx = df.index[df["Level1_Label"]=="BENIGN"]
X_ben = X_scaled.loc[ben_idx, iso_cols]

if len(X_ben) > ISO_SAMPLE:
    X_ben = X_ben.sample(ISO_SAMPLE, random_state=RANDOM_STATE)

iso = IsolationForest(n_estimators=200, contamination=ISO_CONTAM, random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_ben)

ben_scores = -iso.decision_function(X_ben)
iso_thr = float(np.quantile(ben_scores, 0.99))

print("Isolation threshold (99% benign):", iso_thr)

joblib.dump(iso, os.path.join(OUTPUT_DIR, "isolation_forest.pkl"), compress=3)
joblib.dump(iso_cols, os.path.join(OUTPUT_DIR, "iso_features.pkl"), compress=3)
joblib.dump(iso_thr, os.path.join(OUTPUT_DIR, "iso_threshold.pkl"), compress=3)

Isolation threshold (99% benign): 1.6622075295244883e-17


['/content/ids_research_outputs/iso_threshold.pkl']

In [ ]:
sample_idx = attack_df.index
if len(sample_idx) > 200000:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(sample_idx, 200000, replace=False)

X_inv = X_scaled.loc[sample_idx, iso_cols]
anom_scores = -iso.decision_function(X_inv)

top_idx = np.argsort(anom_scores)[-min(HDBSCAN_MAX, len(anom_scores)):]
X_hdb = X_inv.iloc[top_idx].copy()

clusterer = hdbscan.HDBSCAN(min_cluster_size=25, min_samples=10)
clusters = clusterer.fit_predict(X_hdb)
print("Cluster counts (top anomalies):")
print(pd.Series(clusters).value_counts().head(15))

G = nx.Graph()

ports = df.loc[X_hdb.index, "Destination Port"].astype(int).values if "Destination Port" in df.columns else None
X_for_l2 = X_scaled.loc[X_hdb.index, l2_features]
fam_probs = l2.predict_proba(X_for_l2)
fam_pred = l2.classes_[np.argmax(fam_probs, axis=1)]

if ports is not None:
    for p, c, f in zip(ports, clusters, fam_pred):
        pnode = f"port:{int(p)}"
        cnode = f"cluster:{int(c)}"
        fnode = f"family:{str(f)}"
        for a,b in [(pnode,cnode),(pnode,fnode),(cnode,fnode)]:
            if G.has_edge(a,b):
                G[a][b]["weight"] += 1
            else:
                G.add_edge(a,b, weight=1)

    print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())
    port_edges = [(u,v,d["weight"]) for u,v,d in G.edges(data=True)
                  if u.startswith("port:") and v.startswith("cluster:")]
    port_edges = sorted(port_edges, key=lambda x: x[2], reverse=True)[:15]
    print("Top port-cluster edges:", port_edges)
else:
    print("Destination Port missing; graph will be limited.")

Cluster counts (top anomalies):
-1     870
 47    717
 43    437
 70    389
 91    381
 78    323
 16    306
 15    291
 62    277
 73    252
 59    231
 71    222
 42    210
 80    201
 69    200
Name: count, dtype: int64
Graph nodes: 136 edges: 273
Top port-cluster edges: [('port:80', 'cluster:-1', 847), ('port:80', 'cluster:47', 717), ('port:80', 'cluster:43', 437), ('port:80', 'cluster:70', 389), ('port:80', 'cluster:91', 381), ('port:80', 'cluster:78', 323), ('port:80', 'cluster:16', 306), ('port:80', 'cluster:15', 291), ('port:80', 'cluster:62', 277), ('port:80', 'cluster:73', 252), ('port:80', 'cluster:59', 231), ('port:80', 'cluster:71', 222), ('port:80', 'cluster:42', 210), ('port:80', 'cluster:80', 201), ('port:80', 'cluster:69', 200)]


In [ ]:
def decide_action(p_attack, fam_pred, fam_conf, anom_score,
                  l1_thr, fam_thr=0.70, anom_thr=None):
    if anom_thr is None:
        anom_thr = iso_thr

    if p_attack < l1_thr and anom_score < anom_thr:
        return "ALLOW", "BENIGN"

    if fam_conf >= fam_thr:
        if fam_pred in ["DoS","PortScan"]:
            return "BLOCK_OR_RATE_LIMIT", fam_pred
        if fam_pred in ["Bot","BruteForce","WebAttack"]:
            return "ALERT_AND_REVIEW", fam_pred

    if anom_score >= anom_thr:
        return "ESCALATE", "TrueRare_or_Unknown"

    return "REVIEW", fam_pred

PLAYBOOKS = {
    "DoS": "BLOCK_OR_RATE_LIMIT",
    "PortScan": "BLOCK_OR_RATE_LIMIT",
    "Bot": "ALERT_AND_REVIEW",
    "BruteForce": "ALERT_AND_REVIEW",
    "WebAttack": "ALERT_AND_REVIEW",
    "TrueRare_or_Unknown": "ESCALATE",
    "BENIGN": "ALLOW"
}
print("Playbooks:", PLAYBOOKS)

class SimpleBandit:
    def __init__(self, actions):
        self.actions = actions
        self.counts = {a:0 for a in actions}
        self.rewards = {a:0.0 for a in actions}

    def choose(self):
        eps = 0.1
        if np.random.rand() < eps:
            return np.random.choice(self.actions)
        avg = {a: self.rewards[a]/max(1,self.counts[a]) for a in self.actions}
        return max(avg, key=avg.get)

    def update(self, action, reward):
        self.counts[action] += 1
        self.rewards[action] += reward

bandit = SimpleBandit(list(set(PLAYBOOKS.values())))
print("Bandit ready (optional).")

Playbooks: {'DoS': 'BLOCK_OR_RATE_LIMIT', 'PortScan': 'BLOCK_OR_RATE_LIMIT', 'Bot': 'ALERT_AND_REVIEW', 'BruteForce': 'ALERT_AND_REVIEW', 'WebAttack': 'ALERT_AND_REVIEW', 'TrueRare_or_Unknown': 'ESCALATE', 'BENIGN': 'ALLOW'}
Bandit ready (optional).


In [ ]:
def psi(expected, actual, buckets=10):
    expected = np.asarray(expected, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    breakpoints = np.unique(np.quantile(expected, np.linspace(0,1,buckets+1)))
    if len(breakpoints) < 2:
        return 0.0
    e_cnt, _ = np.histogram(expected, bins=breakpoints)
    a_cnt, _ = np.histogram(actual, bins=breakpoints)
    e = np.where(e_cnt == 0, 1e-6, e_cnt/len(expected))
    a = np.where(a_cnt == 0, 1e-6, a_cnt/len(actual))
    return float(np.sum((a-e)*np.log(a/e)))

def drift_table(train_idx, test_idx, features, top_n=12):
    rows = []
    for f in features[:top_n]:
        tr = X_scaled.loc[train_idx, f].values
        te = X_scaled.loc[test_idx, f].values
        psi_val = psi(tr, te, buckets=10)
        ks_stat, ks_p = ks_2samp(tr, te)
        rows.append([f, psi_val, ks_stat, ks_p])
    return pd.DataFrame(rows, columns=["feature","psi","ks_stat","ks_pvalue"]).sort_values("psi", ascending=False)

In [ ]:
def file_order_key(name: str):
    s = str(name).lower()
    day_map = {"monday":0, "tuesday":1, "wednesday":2, "thursday":3, "friday":4}
    day = next((day_map[d] for d in day_map if d in s), 99)
    if "morning" in s: tod = 0
    elif "workinghours" in s: tod = 1
    elif "afternoon" in s: tod = 2
    else: tod = 9
    special = 0
    if "webattacks" in s: special = -1
    if "infilteration" in s or "infiltration" in s: special = 1
    return (day, tod, special, s)

files_sorted = sorted(df[SOURCE_COL].unique().tolist(), key=file_order_key)

print("Files in order:")
for f in files_sorted:
    sub = df[df[SOURCE_COL]==f]
    print(f"- {f} | rows={len(sub)} | attacks={(sub['Level1_Label']=='ATTACK').sum()}")

Files in order:
- Monday-WorkingHours.pcap_ISCX.csv | rows=502787 | attacks=0
- Tuesday-WorkingHours.pcap_ISCX.csv | rows=410197 | attacks=9152
- Wednesday-workingHours.pcap_ISCX.csv | rows=590765 | attacks=193759
- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv | rows=156689 | attacks=2143
- Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv | rows=246030 | attacks=36
- Friday-WorkingHours-Morning.pcap_ISCX.csv | rows=180193 | attacks=1953
- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv | rows=223112 | attacks=128016
- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv | rows=212589 | attacks=90819


In [ ]:
def loo_eval_L1(files, features, seed=42):
    rows = []
    for holdout in files:
        train_idx = df.index[df[SOURCE_COL] != holdout]
        test_idx  = df.index[df[SOURCE_COL] == holdout]

        y_train = df.loc[train_idx, "Level1_Label"]
        y_test  = df.loc[test_idx, "Level1_Label"]

        if (y_test=="ATTACK").sum() == 0:
            rows.append({
                "holdout_file": holdout,
                "rows": len(test_idx),
                "attack_rate_true": 0.0,
                "auc": np.nan,
                "f1_attack": np.nan,
                "note": "No ATTACK in test file"
            })
            continue

        X_train = X_scaled.loc[train_idx, features]
        X_test  = X_scaled.loc[test_idx, features]

        m = lgb.LGBMClassifier(objective="binary", n_estimators=250, random_state=seed, n_jobs=-1, verbosity=-1)
        m.fit(X_train, y_train)

        atk_i = list(m.classes_).index("ATTACK")
        prob = m.predict_proba(X_test)[:, atk_i]
        pred = np.where(prob >= 0.5, "ATTACK", "BENIGN")

        y_bin = (y_test=="ATTACK").astype(int).values
        p_bin = (pred=="ATTACK").astype(int)

        rows.append({
            "holdout_file": holdout,
            "rows": len(test_idx),
            "attack_rate_true": float(y_bin.mean()),
            "attack_rate_pred": float(p_bin.mean()),
            "auc": float(roc_auc_score(y_bin, prob)),
            "precision_attack": float(precision_score(y_bin, p_bin, zero_division=0)),
            "recall_attack": float(recall_score(y_bin, p_bin, zero_division=0)),
            "f1_attack": float(f1_score(y_bin, p_bin, zero_division=0)),
            "note": ""
        })

    return pd.DataFrame(rows)

loo_df = loo_eval_L1(files_sorted, l1_features, seed=RANDOM_STATE)
print(loo_df)

loo_df.to_csv(os.path.join(OUTPUT_DIR, "loo_l1_results.csv"), index=False)

                                        holdout_file    rows  \
0                  Monday-WorkingHours.pcap_ISCX.csv  502787   
1                 Tuesday-WorkingHours.pcap_ISCX.csv  410197   
2               Wednesday-workingHours.pcap_ISCX.csv  590765   
3  Thursday-WorkingHours-Morning-WebAttacks.pcap_...  156689   
4  Thursday-WorkingHours-Afternoon-Infilteration....  246030   
5          Friday-WorkingHours-Morning.pcap_ISCX.csv  180193   
6   Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  223112   
7  Friday-WorkingHours-Afternoon-PortScan.pcap_IS...  212589   

   attack_rate_true       auc  f1_attack                    note  \
0          0.000000       NaN        NaN  No ATTACK in test file   
1          0.022311  0.842201   0.004297                           
2          0.327980  0.666820   0.674453                           
3          0.013677  0.950260   0.025562                           
4          0.000146  0.303524   0.000000                           
5          0.01

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

SUP_FAMS = ["DoS","PortScan","BruteForce","WebAttack","Bot"]

def time_eval_L2(files_sorted, l2_features, seed=42):
    rows = []

    for i in range(len(files_sorted)-2):
        train_files = files_sorted[:i+1]
        val_file = files_sorted[i+1]
        test_file = files_sorted[i+2]

        tr_idx = df.index[(df["_source_file"].isin(train_files)) & (df["Level1_Label"]=="ATTACK") & (df["Level2_Family"].isin(SUP_FAMS))]
        va_idx = df.index[(df["_source_file"]==val_file) & (df["Level1_Label"]=="ATTACK") & (df["Level2_Family"].isin(SUP_FAMS))]
        te_idx = df.index[(df["_source_file"]==test_file) & (df["Level1_Label"]=="ATTACK") & (df["Level2_Family"].isin(SUP_FAMS))]

        if len(tr_idx) < 1000 or len(te_idx) < 100:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "train_attack_rows": len(tr_idx),
                "test_attack_rows": len(te_idx),
                "note": "Too few samples; skipped"
            })
            continue

        y_tr = df.loc[tr_idx, "Level2_Family"]
        y_va = df.loc[va_idx, "Level2_Family"]
        y_te = df.loc[te_idx, "Level2_Family"]

        X_tr = X_scaled.loc[tr_idx, l2_features]
        X_va = X_scaled.loc[va_idx, l2_features] if len(va_idx) > 0 else None
        X_te = X_scaled.loc[te_idx, l2_features]

        classes = np.unique(y_tr)
        if len(classes) < 2:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "train_attack_rows": len(tr_idx),
                "test_attack_rows": len(te_idx),
                "note": "Train has <2 classes; skipped"
            })
            continue

        w = compute_class_weight("balanced", classes=classes, y=y_tr)
        cw2 = dict(zip(classes, w))

        base = lgb.LGBMClassifier(
            objective="multiclass",
            n_estimators=250, learning_rate=0.05,
            num_leaves=31, min_child_samples=50,
            subsample=0.8, colsample_bytree=0.8,
            class_weight=cw2,
            random_state=seed, n_jobs=-1, verbosity=-1
        )
        base.fit(X_tr, y_tr)

        model = base
        calibrated = False
        if X_va is not None and len(y_va.unique()) >= 2 and len(va_idx) > 200:
            cal = CalibratedClassifierCV(base, method="isotonic", cv="prefit")
            cal.fit(X_va, y_va)
            model = cal
            calibrated = True

        pred = model.predict(X_te)

        macro_f1 = f1_score(y_te, pred, average="macro", zero_division=0)
        weighted_f1 = f1_score(y_te, pred, average="weighted", zero_division=0)

        rows.append({
            "train_end": train_files[-1],
            "val_file": val_file,
            "test_file": test_file,
            "train_attack_rows": len(tr_idx),
            "test_attack_rows": len(te_idx),
            "classes_in_train": ",".join(sorted(classes)),
            "calibrated": calibrated,
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "note": ""
        })

    return pd.DataFrame(rows)

time_l2 = time_eval_L2(files_sorted, l2_features, seed=RANDOM_STATE)
time_l2.to_csv(os.path.join(OUTPUT_DIR, "time_like_l2_results.csv"), index=False)
time_l2

,train_end,val_file,test_file,train_attack_rows,test_attack_rows,note,classes_in_train,calibrated,macro_f1,weighted_f1
0,Monday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv,0,193748,Too few samples; skipped,NaN,NaN,NaN,NaN
1,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,9152,2143,Train has <2 classes; skipped,NaN,NaN,NaN,NaN
2,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday-WorkingHours-Afternoon-Infilteration....,202900,0,Too few samples; skipped,NaN,NaN,NaN,NaN
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday-WorkingHours-Afternoon-Infilteration....,Friday-WorkingHours-Morning.pcap_ISCX.csv,205043,1953,,"BruteForce,DoS,WebAttack",False,0.0,0.0
4,Thursday-WorkingHours-Afternoon-Infilteration....,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,205043,128016,,"BruteForce,DoS,WebAttack",False,1.0,1.0
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,206996,90819,,"Bot,BruteForce,DoS,WebAttack",False,0.0,0.0


In [ ]:
import lightgbm as lgb

def tune_threshold_on_val(y_val_bin, val_prob, target_recall=0.95):
    prec, rec, thr = precision_recall_curve(y_val_bin, val_prob)
    valid = np.where(rec >= target_recall)[0]
    if len(valid) == 0:
        return 0.5
    best_i = valid[np.argmax(prec[valid])]
    return float(thr[best_i-1] if best_i > 0 else 0.0)

def eval_time_split_L1(train_files, val_file, test_file, features, target_recall=0.95, seed=42):
    # indices
    tr_idx = df.index[df["_source_file"].isin(train_files)]
    va_idx = df.index[df["_source_file"] == val_file]
    te_idx = df.index[df["_source_file"] == test_file]

    # data
    X_tr = X_scaled.loc[tr_idx, features]
    y_tr = df.loc[tr_idx, "Level1_Label"]

    X_va = X_scaled.loc[va_idx, features]
    y_va = df.loc[va_idx, "Level1_Label"]

    X_te = X_scaled.loc[te_idx, features]
    y_te = df.loc[te_idx, "Level1_Label"]

    if (y_va == "ATTACK").sum() == 0 or (y_va == "BENIGN").sum() == 0:
        return {"error": f"Validation file {val_file} has only one class; pick a different val file."}

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=100,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(X_tr, y_tr)

    attack_idx = list(model.classes_).index("ATTACK")

    va_prob = model.predict_proba(X_va)[:, attack_idx]
    y_va_bin = (y_va == "ATTACK").astype(int).values
    thr = tune_threshold_on_validation(y_va_bin, va_prob, target_recall=target_recall)

    te_prob = model.predict_proba(X_te)[:, attack_idx]
    te_pred = np.where(te_prob >= thr, "ATTACK", "BENIGN")

    y_te_bin = (y_te == "ATTACK").astype(int).values
    p_te_bin = (te_pred == "ATTACK").astype(int)

    # metrics
    out = {
        "train_files": " | ".join(train_files),
        "val_file": val_file,
        "test_file": test_file,
        "test_rows": len(te_idx),
        "test_attack_rate_true": float(y_te_bin.mean()),
        "test_attack_rate_pred": float(p_te_bin.mean()),
        "threshold": float(thr),
        "auc_test": float(roc_auc_score(y_te_bin, te_prob)) if len(np.unique(y_te_bin))>1 else np.nan,
        "precision_attack": float(precision_score(y_te_bin, p_te_bin, zero_division=0)),
        "recall_attack": float(recall_score(y_te_bin, p_te_bin, zero_division=0)),
        "f1_attack": float(f1_score(y_te_bin, p_te_bin, zero_division=0)),
    }

    cm = confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"])
    out["cm_TN_FP_FN_TP"] = cm.ravel().tolist()
    return out

In [ ]:
worst_file = "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"

te_idx = df.index[(df["_source_file"]==worst_file) &
                  (df["Level1_Label"]=="ATTACK") &
                  (df["Level2_Family"].isin(SUP_FAMS))]

tr_idx = df.index[(df["_source_file"]!=worst_file) &
                  (df["Level1_Label"]=="ATTACK") &
                  (df["Level2_Family"].isin(SUP_FAMS))]

X_tr = X_scaled.loc[tr_idx, l2_features]
y_tr = df.loc[tr_idx, "Level2_Family"]
X_te = X_scaled.loc[te_idx, l2_features]
y_te = df.loc[te_idx, "Level2_Family"]

classes = np.unique(y_tr)
w = compute_class_weight("balanced", classes=classes, y=y_tr)
cw = dict(zip(classes, w))

m = lgb.LGBMClassifier(objective="multiclass", n_estimators=300, learning_rate=0.05,
                       num_leaves=31, min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
                       class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
m.fit(X_tr, y_tr)
pred = m.predict(X_te)

print("True label counts:", y_te.value_counts())
print("Pred label counts:", pd.Series(pred).value_counts())
print(classification_report(y_te, pred, zero_division=0))

True label counts: Level2_Family
PortScan    90819
Name: count, dtype: int64
Pred label counts: Bot           87556
BruteForce     2700
DoS             442
WebAttack       121
Name: count, dtype: int64
              precision    recall  f1-score   support

         Bot       0.00      0.00      0.00       0.0
  BruteForce       0.00      0.00      0.00       0.0
         DoS       0.00      0.00      0.00       0.0
    PortScan       0.00      0.00      0.00   90819.0
   WebAttack       0.00      0.00      0.00       0.0

    accuracy                           0.00   90819.0
   macro avg       0.00      0.00      0.00   90819.0
weighted avg       0.00      0.00      0.00   90819.0



In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, precision_recall_curve, precision_score, recall_score, f1_score, confusion_matrix

def time_eval_L1_all_windows(files_sorted, features, target_recall=0.95, seed=42):
    rows = []

    for i in range(len(files_sorted)-2):
        train_files = files_sorted[:i+1]
        val_file = files_sorted[i+1]
        test_file = files_sorted[i+2]

        tr_idx = df.index[df["_source_file"].isin(train_files)]
        va_idx = df.index[df["_source_file"] == val_file]
        te_idx = df.index[df["_source_file"] == test_file]

        y_tr = df.loc[tr_idx, "Level1_Label"]
        y_va = df.loc[va_idx, "Level1_Label"]
        y_te = df.loc[te_idx, "Level1_Label"]

        if y_tr.nunique() < 2:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "TRAIN single-class; skipped"
            })
            continue

        if y_va.nunique() < 2:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "VAL single-class; skipped"
            })
            continue

        X_tr = X_scaled.loc[tr_idx, features]
        X_va = X_scaled.loc[va_idx, features]
        X_te = X_scaled.loc[te_idx, features]

        m = lgb.LGBMClassifier(
            objective="binary", n_estimators=250,
            learning_rate=0.05, num_leaves=31,
            min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
            random_state=seed, n_jobs=-1, verbosity=-1
        )
        m.fit(X_tr, y_tr)

        if "ATTACK" not in m.classes_:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "Model trained without ATTACK class; skipped"
            })
            continue

        atk_i = list(m.classes_).index("ATTACK")

        va_prob = m.predict_proba(X_va)[:, atk_i]
        thr = tune_threshold_on_val((y_va=="ATTACK").astype(int).values, va_prob, target_recall=target_recall)

        te_prob = m.predict_proba(X_te)[:, atk_i]
        te_pred = np.where(te_prob >= thr, "ATTACK", "BENIGN")

        y_bin = (y_te=="ATTACK").astype(int).values
        p_bin = (te_pred=="ATTACK").astype(int)

        row = {
            "train_end": train_files[-1],
            "val_file": val_file,
            "test_file": test_file,
            "test_rows": len(te_idx),
            "test_attack_rate_true": float(y_bin.mean()),
            "test_attack_rate_pred": float(p_bin.mean()),
            "threshold": float(thr),
            "note": ""
        }

        if y_bin.sum() > 0 and (y_bin==0).sum() > 0:
            row.update({
                "auc_test": float(roc_auc_score(y_bin, te_prob)),
                "precision_attack": float(precision_score(y_bin, p_bin, zero_division=0)),
                "recall_attack": float(recall_score(y_bin, p_bin, zero_division=0)),
                "f1_attack": float(f1_score(y_bin, p_bin, zero_division=0)),
                "cm_TN_FP_FN_TP": confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"]).ravel().tolist()
            })
        else:
            row.update({
                "auc_test": np.nan,
                "precision_attack": float(precision_score(y_bin, p_bin, zero_division=0)),
                "recall_attack": float(recall_score(y_bin, p_bin, zero_division=0)),
                "f1_attack": float(f1_score(y_bin, p_bin, zero_division=0)),
                "cm_TN_FP_FN_TP": confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"]).ravel().tolist(),
                "note": "AUC undefined (single-class test)"
            })

        rows.append(row)

    return pd.DataFrame(rows)

time_l1_all = time_eval_L1_all_windows(files_sorted, l1_features, target_recall=0.95, seed=RANDOM_STATE)
time_l1_all.to_csv(os.path.join(OUTPUT_DIR, "time_like_l1_all_windows.csv"), index=False)
time_l1_all

,train_end,val_file,test_file,note,test_rows,test_attack_rate_true,test_attack_rate_pred,threshold,auc_test,precision_attack,recall_attack,f1_attack,cm_TN_FP_FN_TP
0,Monday-WorkingHours.pcap_ISCX.csv,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv,TRAIN single-class; skipped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tuesday-WorkingHours.pcap_ISCX.csv,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,,156689.0,0.013677,0.495510,3.734185e-08,0.977328,0.027601,1.000000,0.053720,"[79048, 75498, 0, 2143]"
2,Wednesday-workingHours.pcap_ISCX.csv,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday-WorkingHours-Afternoon-Infilteration....,,246030.0,0.000146,0.007434,7.482066e-04,0.845571,0.000000,0.000000,0.000000,"[244165, 1829, 36, 0]"
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Thursday-WorkingHours-Afternoon-Infilteration....,Friday-WorkingHours-Morning.pcap_ISCX.csv,,180193.0,0.010838,0.295811,1.826405e-06,0.698874,0.014427,0.393753,0.027834,"[125706, 52534, 1184, 769]"
4,Thursday-WorkingHours-Afternoon-Infilteration....,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,,223112.0,0.573775,0.954565,5.994016e-07,0.998475,0.601075,0.999984,0.750835,"[10135, 84961, 2, 128014]"
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,,212589.0,0.427205,0.066057,9.288452e-04,0.935449,0.767001,0.118599,0.205432,"[118498, 3272, 80048, 10771]"


In [ ]:
def time_eval_L1(files, features, target_recall=0.95, seed=42):
    rows = []

    for i in range(len(files)-2):
        train_files = files[:i+1]
        val_file = files[i+1]
        test_file = files[i+2]

        tr_idx = df.index[df[SOURCE_COL].isin(train_files)]
        va_idx = df.index[df[SOURCE_COL] == val_file]
        te_idx = df.index[df[SOURCE_COL] == test_file]

        y_tr = df.loc[tr_idx, "Level1_Label"]
        y_va = df.loc[va_idx, "Level1_Label"]
        y_te = df.loc[te_idx, "Level1_Label"]

        if y_tr.nunique() < 2:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "TRAIN single-class; skipped"
            })
            continue

        if y_va.nunique() < 2:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "VAL single-class; skipped"
            })
            continue

        X_tr = X_scaled.loc[tr_idx, features]
        X_va = X_scaled.loc[va_idx, features]
        X_te = X_scaled.loc[te_idx, features]

        m = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=250,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1
        )
        m.fit(X_tr, y_tr)

        if "ATTACK" not in m.classes_:
            rows.append({
                "train_end": train_files[-1],
                "val_file": val_file,
                "test_file": test_file,
                "note": "Model trained without ATTACK class; skipped"
            })
            continue

        atk_i = list(m.classes_).index("ATTACK")

        va_prob = m.predict_proba(X_va)[:, atk_i]
        y_va_bin = (y_va=="ATTACK").astype(int).values
        thr = tune_threshold_on_val(y_va_bin, va_prob, target_recall=target_recall)

        te_prob = m.predict_proba(X_te)[:, atk_i]
        te_pred = np.where(te_prob >= thr, "ATTACK", "BENIGN")

        y_bin = (y_te=="ATTACK").astype(int).values
        p_bin = (te_pred=="ATTACK").astype(int)

        row = {
            "train_end": train_files[-1],
            "val_file": val_file,
            "test_file": test_file,
            "test_rows": len(te_idx),
            "test_attack_rate_true": float(y_bin.mean()),
            "test_attack_rate_pred": float(p_bin.mean()),
            "threshold": float(thr),
            "note": ""
        }

        if y_bin.sum() > 0 and (y_bin==0).sum() > 0:
            row.update({
                "auc_test": float(roc_auc_score(y_bin, te_prob)),
                "precision_attack": float(precision_score(y_bin, p_bin, zero_division=0)),
                "recall_attack": float(recall_score(y_bin, p_bin, zero_division=0)),
                "f1_attack": float(f1_score(y_bin, p_bin, zero_division=0)),
                "cm_TN_FP_FN_TP": confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"]).ravel().tolist()
            })
        else:
            row.update({
                "auc_test": np.nan,
                "precision_attack": float(precision_score(y_bin, p_bin, zero_division=0)),
                "recall_attack": float(recall_score(y_bin, p_bin, zero_division=0)),
                "f1_attack": float(f1_score(y_bin, p_bin, zero_division=0)),
                "cm_TN_FP_FN_TP": confusion_matrix(y_te, te_pred, labels=["BENIGN","ATTACK"]).ravel().tolist(),
                "note": "AUC undefined (single-class test)"
            })

        rows.append(row)

    return pd.DataFrame(rows)

time_df = time_eval_L1(files_sorted, l1_features, target_recall=0.95, seed=RANDOM_STATE)
print(time_df)
time_df.to_csv(os.path.join(OUTPUT_DIR, "time_like_l1_results.csv"), index=False)

                                           train_end  \
0                  Monday-WorkingHours.pcap_ISCX.csv   
1                 Tuesday-WorkingHours.pcap_ISCX.csv   
2               Wednesday-workingHours.pcap_ISCX.csv   
3  Thursday-WorkingHours-Morning-WebAttacks.pcap_...   
4  Thursday-WorkingHours-Afternoon-Infilteration....   
5          Friday-WorkingHours-Morning.pcap_ISCX.csv   

                                            val_file  \
0                 Tuesday-WorkingHours.pcap_ISCX.csv   
1               Wednesday-workingHours.pcap_ISCX.csv   
2  Thursday-WorkingHours-Morning-WebAttacks.pcap_...   
3  Thursday-WorkingHours-Afternoon-Infilteration....   
4          Friday-WorkingHours-Morning.pcap_ISCX.csv   
5   Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   

                                           test_file  \
0               Wednesday-workingHours.pcap_ISCX.csv   
1  Thursday-WorkingHours-Morning-WebAttacks.pcap_...   
2  Thursday-WorkingHours-Afternoon-Infilterati

In [ ]:
# Optimistic L1 is your row-random split metrics (already printed earlier).
# Now summarize Realistic:
loo_summary = loo_df.copy()
loo_summary["evaluation"] = "Realistic_LOO"
loo_summary.to_csv(os.path.join(OUTPUT_DIR, "realistic_loo_summary.csv"), index=False)

time_summary = time_l1_all.copy()
time_summary["evaluation"] = "Realistic_TimeLike"
time_summary.to_csv(os.path.join(OUTPUT_DIR, "realistic_timelike_summary.csv"), index=False)

print("Saved:")
print("-", os.path.join(OUTPUT_DIR, "realistic_loo_summary.csv"))
print("-", os.path.join(OUTPUT_DIR, "realistic_timelike_summary.csv"))
print("-", os.path.join(OUTPUT_DIR, "time_like_l2_results.csv"))

Saved:
- /content/ids_research_outputs/realistic_loo_summary.csv
- /content/ids_research_outputs/realistic_timelike_summary.csv
- /content/ids_research_outputs/time_like_l2_results.csv


In [ ]:
worst_file = "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"

te_idx = df.index[(df["_source_file"]==worst_file) &
                  (df["Level1_Label"]=="ATTACK") &
                  (df["Level2_Family"].isin(SUP_FAMS))]

tr_idx = df.index[(df["_source_file"]!=worst_file) &
                  (df["Level1_Label"]=="ATTACK") &
                  (df["Level2_Family"].isin(SUP_FAMS))]

X_tr = X_scaled.loc[tr_idx, l2_features]
y_tr = df.loc[tr_idx, "Level2_Family"]
X_te = X_scaled.loc[te_idx, l2_features]
y_te = df.loc[te_idx, "Level2_Family"]

classes = np.unique(y_tr)
w = compute_class_weight("balanced", classes=classes, y=y_tr)
cw = dict(zip(classes, w))

m = lgb.LGBMClassifier(objective="multiclass", n_estimators=300, learning_rate=0.05,
                       num_leaves=31, min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
                       class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)
m.fit(X_tr, y_tr)
pred = m.predict(X_te)

print("True label counts:", y_te.value_counts())
print("Pred label counts:", pd.Series(pred).value_counts())
print(classification_report(y_te, pred, zero_division=0))

True label counts: Level2_Family
PortScan    90819
Name: count, dtype: int64
Pred label counts: Bot           87556
BruteForce     2700
DoS             442
WebAttack       121
Name: count, dtype: int64
              precision    recall  f1-score   support

         Bot       0.00      0.00      0.00       0.0
  BruteForce       0.00      0.00      0.00       0.0
         DoS       0.00      0.00      0.00       0.0
    PortScan       0.00      0.00      0.00   90819.0
   WebAttack       0.00      0.00      0.00       0.0

    accuracy                           0.00   90819.0
   macro avg       0.00      0.00      0.00   90819.0
weighted avg       0.00      0.00      0.00   90819.0



In [ ]:
print("Total PortScan in full df:", (df["Level2_Family"]=="PortScan").sum())
print(df[df["Level2_Family"]=="PortScan"]["_source_file"].value_counts().head(10))

Total PortScan in full df: 90819
_source_file
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    90819
Name: count, dtype: int64


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import lightgbm as lgb

SUP_FAMS = ["DoS","PortScan","BruteForce","WebAttack","Bot"]

def in_file_family_eval(df, X_scaled, features, test_file, seed=42):
    sub_idx = df.index[(df["_source_file"]==test_file) &
                       (df["Level1_Label"]=="ATTACK") &
                       (df["Level2_Family"].isin(SUP_FAMS))]
    if len(sub_idx) == 0:
        return None

    y = df.loc[sub_idx, "Level2_Family"]
    X = X_scaled.loc[sub_idx, features]

    # Stratified split inside same file
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    classes = np.unique(y_tr)
    if len(classes) < 2:
        return {
            "file": test_file,
            "note": "Only one family present in file; multi-class not definable within file",
            "rows": len(sub_idx),
            "classes": ",".join(classes)
        }

    model = lgb.LGBMClassifier(
        objective="multiclass",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    macro_f1 = f1_score(y_te, pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_te, pred, average="weighted", zero_division=0)

    return {
        "file": test_file,
        "rows": len(sub_idx),
        "classes": ",".join(sorted(np.unique(y))),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "note": ""
    }

files = sorted(df["_source_file"].unique())
rows = []
for f in files:
    out = in_file_family_eval(df, X_scaled, l2_features, f, seed=RANDOM_STATE)
    if out is not None:
        rows.append(out)

infile_l2_df = pd.DataFrame(rows)
infile_l2_df

,file,note,rows,classes
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,Only one family present in file; multi-class n...,128016,DoS
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,Only one family present in file; multi-class n...,90819,PortScan
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,Only one family present in file; multi-class n...,1953,Bot
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,Only one family present in file; multi-class n...,2143,WebAttack
4,Tuesday-WorkingHours.pcap_ISCX.csv,Only one family present in file; multi-class n...,9152,BruteForce
5,Wednesday-workingHours.pcap_ISCX.csv,Only one family present in file; multi-class n...,193748,DoS


In [ ]:
infile_l2_df['note'][1]

'Only one family present in file; multi-class not definable within file'

In [ ]:
print(df[df["Level2_Family"]=="PortScan"]["_source_file"].value_counts())

_source_file
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    90819
Name: count, dtype: int64
